# BrandMate 밈 광고 문구 평가 — Google Colab

이 노트북은 저장소를 clone하고, 같은 Colab GPU에서 Qwen2.5-7B AWQ를 vLLM으로 실행한 뒤 기존 밈 평가 runner를 호출합니다.

시작 전에 런타임 유형을 **T4 GPU**로 바꾸고, GPT Judge를 사용할 경우 Colab Secrets에 `BRANDMATE_OPENAI_API_KEY`를 등록해 주세요. 키를 코드 셀이나 `.env`에 직접 입력하지 마세요.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys
import time
import urllib.request

REPO_URL = "https://github.com/imella0706/final_1_team.git"
BRANCH = "feat/reflect_meme"  # main에 병합했다면 "main"으로 변경
REPO_DIR = Path("/content/final_1_team")
API_DIR = REPO_DIR / "apps" / "api"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct-AWQ"
VLLM_VERSION = "0.25.1"
VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
MAX_MODEL_LEN = 12288
RUN_JUDGE = True
OUTPUT_DIR = Path("/content/drive/MyDrive/BrandMate/evaluations/meme")

def run_command(command, *, cwd=None, env=None):
    command = [str(part) for part in command]
    print("+", shlex.join(command))
    return subprocess.run(command, cwd=cwd, env=env, check=True)

try:
    gpu_info = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total",
            "--format=csv,noheader",
        ],
        text=True,
    ).strip()
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    raise RuntimeError(
        "GPU가 없습니다. Colab의 런타임 유형을 T4 GPU로 변경한 뒤 다시 실행하세요."
    ) from error
print("GPU:", gpu_info)

## 1. 저장소 clone 및 필수 파일 확인

이미 clone한 런타임에서 다시 실행하면 현재 브랜치를 fast-forward 방식으로 갱신합니다. 로컬 PC에서 커밋·push하지 않은 파일은 clone되지 않습니다.

In [ ]:
if (REPO_DIR / ".git").is_dir():
    run_command(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    run_command(["git", "-C", REPO_DIR, "checkout", BRANCH])
    run_command(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH])
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR}가 있지만 Git 저장소가 아닙니다. 경로를 비우거나 REPO_DIR를 바꾸세요.")
else:
    run_command(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])

required_files = [
    "apps/api/scripts/evaluate_meme_arms.py",
    "apps/api/app/evaluation/meme_arm_runner.py",
    "apps/api/app/evaluation/text_judge.py",
    "apps/api/evals/meme_5arm_experiment.json",
    "apps/api/evals/meme_ad_copy_cases.json",
    "apps/api/evals/few_shot_examples.json",
    "apps/api/evals/meme_fixture_review.json",
    "gather_data/trendcard.json",
]
missing = [relative for relative in required_files if not (REPO_DIR / relative).is_file()]
if missing:
    raise RuntimeError(
        "clone한 브랜치에 평가 필수 파일이 없습니다. 로컬 변경을 커밋·push했는지 확인하세요:\n- "
        + "\n- ".join(missing)
    )
print(f"필수 파일 {len(required_files)}개 확인 완료: {REPO_DIR}")

## 2. vLLM과 프로젝트 의존성 설치

GPU용 패키지 설치에는 몇 분이 걸릴 수 있습니다. 이 셀보다 앞에서 `torch`를 import하지 않습니다.

In [ ]:
run_command([sys.executable, "-m", "pip", "install", "-q", "uv"])
uv = shutil.which("uv")
if not uv:
    raise RuntimeError("uv 설치 후 실행 파일을 찾지 못했습니다.")
run_command([uv, "pip", "install", "--system", "--torch-backend=auto", f"vllm=={VLLM_VERSION}"])
run_command([uv, "pip", "install", "--system", "-e", API_DIR])
print("의존성 설치 완료")

## 3. Google Drive와 Secrets 연결

모델 캐시는 휘발성 `/content`에 두고 평가 결과만 Drive에 저장합니다. `RUN_JUDGE=False`이면 OpenAI 키 없이 생성 결과만 확인합니다.

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def read_secret(name, *, required=False):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab Secrets에 {name}를 등록하고 노트북 액세스를 허용하세요."
        )
    return value

openai_key = read_secret("BRANDMATE_OPENAI_API_KEY", required=RUN_JUDGE)
hf_token = read_secret("HF_TOKEN")
if openai_key:
    os.environ["BRANDMATE_OPENAI_API_KEY"] = openai_key
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
os.environ["HF_HOME"] = "/content/huggingface"
print("Drive 및 Secret 설정 완료 (키 값은 출력하지 않음)")

## 4. Qwen AWQ vLLM 서버 시작

서버가 이미 같은 모델을 제공 중이면 재사용합니다. 최초 실행에는 모델 다운로드 시간이 필요합니다. 실패 로그는 `/content/vllm-qwen.log`에 남습니다.

In [ ]:
VLLM_LOG = Path("/content/vllm-qwen.log")
MODELS_URL = f"{VLLM_BASE_URL}/models"

def served_model_ids():
    try:
        with urllib.request.urlopen(MODELS_URL, timeout=3) as response:
            payload = json.loads(response.read().decode("utf-8"))
        return {item.get("id") for item in payload.get("data", [])}
    except Exception:
        return set()

models = served_model_ids()
if models and MODEL_ID not in models:
    raise RuntimeError(f"8000 포트가 다른 모델 서버에 사용 중입니다: {sorted(models)}")

if MODEL_ID not in models:
    vllm = shutil.which("vllm")
    if not vllm:
        raise RuntimeError("vllm 실행 파일을 찾지 못했습니다. 의존성 설치 셀을 다시 실행하세요.")
    command = [
        vllm,
        "serve",
        MODEL_ID,
        "--served-model-name",
        MODEL_ID,
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--dtype",
        "half",
        "--quantization",
        "awq",
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        "0.90",
        "--max-num-seqs",
        "1",
        "--enforce-eager",
    ]
    print("vLLM 시작:", shlex.join(command))
    vllm_log_handle = VLLM_LOG.open("w", encoding="utf-8")
    vllm_process = subprocess.Popen(
        command,
        cwd="/content",
        env=os.environ.copy(),
        stdout=vllm_log_handle,
        stderr=subprocess.STDOUT,
    )
    deadline = time.monotonic() + 1200
    while time.monotonic() < deadline:
        if vllm_process.poll() is not None:
            tail = VLLM_LOG.read_text(encoding="utf-8", errors="replace")[-5000:]
            raise RuntimeError(f"vLLM이 시작 중 종료됐습니다. 로그 마지막 부분:\n{tail}")
        models = served_model_ids()
        if MODEL_ID in models:
            break
        time.sleep(5)
    else:
        tail = VLLM_LOG.read_text(encoding="utf-8", errors="replace")[-5000:]
        raise TimeoutError(f"20분 안에 vLLM이 준비되지 않았습니다. 로그 마지막 부분:\n{tail}")

print("vLLM 준비 완료:", sorted(served_model_ids()))

## 5. endpoint 설정 및 비용 없는 dry-run

실험 JSON의 기준 모델 ID는 그대로 두고, 실제 호출 모델만 환경변수로 AWQ endpoint에 연결합니다. 평가 runner는 별도 프로세스로 실행되므로 환경변수를 설정한 뒤 호출합니다.

In [ ]:
evaluation_env = os.environ.copy()
evaluation_env.update(
    {
        "BRANDMATE_QWEN_BASE_URL": VLLM_BASE_URL,
        "BRANDMATE_QWEN_MODEL": MODEL_ID,
        "BRANDMATE_QWEN_API_KEY": "local",
        "BRANDMATE_LLM_TIMEOUT_SECONDS": "600",
    }
)
dry_run_command = [
    sys.executable,
    "-m",
    "scripts.evaluate_meme_arms",
    "--dry-run",
    "--case-limit",
    "1",
    "--repeats",
    "1",
]
if not RUN_JUDGE:
    dry_run_command.append("--skip-judge")
run_command(dry_run_command, cwd=API_DIR, env=evaluation_env)

## 6. 네 후보를 1개 케이스로 smoke 평가

활성 후보 `trendcard`, `few_shot_good`, `few_shot_good_bad`, `structured_cot`을 각각 한 번 생성합니다. `RUN_JUDGE=True`이면 생성마다 GPT Judge도 호출합니다. 결과는 Google Drive에 저장됩니다.

In [ ]:
smoke_command = [
    sys.executable,
    "-m",
    "scripts.evaluate_meme_arms",
    "--case-limit",
    "1",
    "--repeats",
    "1",
    "--concurrency",
    "1",
    "--allow-unreviewed-fixtures",
    "--output-dir",
    OUTPUT_DIR,
]
if not RUN_JUDGE:
    smoke_command.append("--skip-judge")
run_command(smoke_command, cwd=API_DIR, env=evaluation_env)

In [ ]:
from IPython.display import Markdown, display

reports = sorted(OUTPUT_DIR.glob("*/report.md"), key=lambda path: path.stat().st_mtime)
if not reports:
    raise RuntimeError(f"report.md를 찾지 못했습니다: {OUTPUT_DIR}")
latest_report = reports[-1]
print("최신 보고서:", latest_report)
display(Markdown(latest_report.read_text(encoding="utf-8")))

## 전체 평가 전 주의

현재 fixture는 검수 전 상태이므로 기본 smoke만 허용됩니다. 사람·권리 검수를 완료한 뒤 전체 평가를 실행할 때는 `--allow-large-run`을 사용하세요. 전체 runner는 모든 trial 종료 후 최종 보고서를 기록하므로 긴 실행 중 Colab이 종료될 수 있다는 점도 고려해야 합니다. 자세한 내용은 `docs/COLAB_MEME_EVALUATION.md`를 확인하세요.